# UK Wind Curtailment Analysis - 2025

**Microsoft Fabric | OneLake | PySpark | Delta Lake | Power BI**

This notebook documents the data ingestion, exploration, quality checks and Lakehouse preparation used in the project. The source code is retained from the working Fabric notebook. Execution outputs, account-specific Fabric metadata and cached previews have been removed for safe, readable publication.

**Data sources:** Elexon June 2025 Bid-Offer Acceptance Level (BOALF) data; Elexon BM unit reference; NESO 2025 historic demand; Renewable Energy Foundation 2025 monthly wind constraints. Files must be placed in the attached Fabric Lakehouse `Files/` folder under the names referenced below.

**Important analytical distinction:** A negative `levelTo - levelFrom` identifies a downward power-level transition in a BOALF segment. It does **not**, on its own, prove physical wind curtailment or quantify curtailed MWh. The REF annual curtailment figures come from a different source and should not be equated with BOALF segment counts.

**Reproducibility:** Requires a Fabric PySpark notebook with the appropriate Lakehouse attached. Running the notebook from top to bottom includes `mode("overwrite")` writes that replace four Delta tables; review destinations before rerunning. The original dashboard was created from the project tables; this published notebook has not been re-executed after cleanup.


In [ ]:
# Read June 2025 balancing acceptance data
df_boa = spark.read.csv(
    "Files/Elexon_BOALF_June_2025.csv",
    header=True,
    inferSchema=True
)

# Display the first five rows
display(df_boa.limit(5))

# Count the records
print("Total records:", df_boa.count())


## 1. Load Elexon BM unit reference data


In [ ]:
# Read Elexon BM Unit Reference Data
df_units_raw = spark.read.option("multiLine", True).json(
    "Files/Elexon_BM_Unit_Reference.json"
)

# Check the structure of the dataset
df_units_raw.printSchema()

# Preview the data
display(df_units_raw.limit(5))

## 2. Identify wind BM units and assess reference data quality


In [ ]:
# Filter the dataset to include only wind units
df_wind = df_units_raw.filter(
    df_units_raw.fuelType == "WIND"
)

# Count wind units
print("Total wind units:", df_wind.count())

# Preview selected columns
display(df_wind.select(
    "bmUnitName",
    "elexonBmUnit",
    "nationalGridBmUnit",
    "fuelType"
).limit(10))

In [ ]:
from pyspark.sql.functions import col

# Check missing values in important columns
df_wind.select(
    *[
        col(c).isNull().cast("int").alias(c)
        for c in ["bmUnitName", "elexonBmUnit", "nationalGridBmUnit"]
    ]
).groupBy().sum().show()

# Check duplicate national grid unit IDs
print(
    "Duplicate unit IDs:",
    df_wind.count() -
    df_wind.select("nationalGridBmUnit").distinct().count()
)

In [ ]:
from pyspark.sql.functions import col, count

# Find duplicated National Grid unit IDs
duplicate_ids = (
    df_wind.groupBy("nationalGridBmUnit")
    .count()
    .filter(col("count") > 1)
)

# Display the complete records for duplicated IDs
display(
    df_wind.join(
        duplicate_ids.select("nationalGridBmUnit"),
        on="nationalGridBmUnit",
        how="inner"
    )
)

In [ ]:
display(
    df_wind.filter(
        col("nationalGridBmUnit") == "WLNYO-4"
    ).select(
        "nationalGridBmUnit",
        "elexonBmUnit",
        "eic",
        "generationCapacity"
    )
)

In [ ]:
# Create a unique wind unit reference table
df_wind_unique = df_wind.select(
    "elexonBmUnit",
    "nationalGridBmUnit",
    "fuelType"
).dropDuplicates(["elexonBmUnit"])

# Check the results
print("Original records:", df_wind.count())
print("Unique records:", df_wind_unique.count())

## 3. Match BOALF acceptance records to wind BM units

The join uses `nationalGridBmUnit`. Verify coverage and reference-data uniqueness before interpreting unmatched records.


In [ ]:
# Display the BOALF dataset column names
print(df_boa.columns)

In [ ]:
# Get unique wind unit IDs
wind_ids = df_wind.select(
    "nationalGridBmUnit"
).distinct()

# Match BOALF records with wind unit IDs
df_wind_boa = df_boa.join(
    wind_ids,
    on="nationalGridBmUnit",
    how="inner"
)

# Count matching records
print("Total BOALF records:", df_boa.count())
print("Wind BOALF records:", df_wind_boa.count())

In [ ]:
# Preview wind balancing acceptance records
display(
    df_wind_boa.select(
        "settlementDate",
        "nationalGridBmUnit",
        "levelFrom",
        "levelTo",
        "timeFrom",
        "timeTo"
    ).limit(10)
)

## 4. Examine changes in accepted power levels

`powerChange = levelTo - levelFrom` is measured in MW (assuming the source fields use MW). A negative value is labelled a **decrease**, not independently verified curtailed generation. Segment counts and MW differences are not MWh.


In [ ]:
from pyspark.sql.functions import col, when

# Calculate the change in power level
df_wind_changes = df_wind_boa.withColumn(
    "powerChange",
    col("levelTo") - col("levelFrom")
)

# Classify the changes
df_wind_changes = df_wind_changes.withColumn(
    "changeType",
    when(col("powerChange") < 0, "Decrease")
    .when(col("powerChange") > 0, "Increase")
    .otherwise("No Change")
)

# Count each type
df_wind_changes.groupBy("changeType").count().show()

In [ ]:
from pyspark.sql.functions import min, max, avg

# Examine records with decreasing power levels
df_decreases = df_wind_changes.filter(
    col("powerChange") < 0
)

# Summarise power reductions
df_decreases.select(
    min("powerChange").alias("Largest_Decrease_MW"),
    max("powerChange").alias("Smallest_Decrease_MW"),
    avg("powerChange").alias("Average_Decrease_MW")
).show()

## 5. Inspect durations and balancing acceptance segments

A single acceptance can have multiple time segments, including decreasing, constant and increasing levels. Do not interpret repeated acceptance numbers alone as duplicate records.


In [ ]:
from pyspark.sql.functions import col, unix_timestamp

# Calculate instruction duration in minutes
df_decreases = df_decreases.withColumn(
    "durationMinutes",
    (
        unix_timestamp(col("timeTo")) -
        unix_timestamp(col("timeFrom"))
    ) / 60
)

# Preview the results
display(
    df_decreases.select(
        "nationalGridBmUnit",
        "powerChange",
        "timeFrom",
        "timeTo",
        "durationMinutes"
    ).limit(10)
)

In [ ]:
from pyspark.sql.functions import min, max, avg, col

# Check instruction durations
df_decreases.select(
    min("durationMinutes").alias("Minimum_Minutes"),
    max("durationMinutes").alias("Maximum_Minutes"),
    avg("durationMinutes").alias("Average_Minutes")
).show()

# Check invalid durations
print(
    "Invalid durations:",
    df_decreases.filter(
        col("durationMinutes").isNull() |
        (col("durationMinutes") <= 0)
    ).count()
)

In [ ]:
# Count records per balancing acceptance
df_decreases.groupBy(
    "settlementDate",
    "nationalGridBmUnit",
    "acceptanceNumber"
).count().orderBy(
    "count", ascending=False
).show(10, truncate=False)

In [ ]:
# Examine all segments of one acceptance
display(
    df_wind_boa.filter(
        (col("settlementDate") == "2025-06-03") &
        (col("nationalGridBmUnit") == "SGRWO-4") &
        (col("acceptanceNumber") == 20247)
    ).select(
        "acceptanceNumber",
        "timeFrom",
        "timeTo",
        "levelFrom",
        "levelTo"
    ).orderBy("timeFrom")
)

In [ ]:
df_wind_boa.filter(
    (col("nationalGridBmUnit") == "SGRWO-4") &
    (col("settlementDate") == "2025-06-03") &
    (col("acceptanceNumber") == 20247)
).select(
    "timeFrom", "timeTo", "levelFrom", "levelTo"
).orderBy("timeFrom").show(20, truncate=False)

In [ ]:
# Examine the flags for acceptance 20247
df_wind_boa.filter(
    (col("nationalGridBmUnit") == "SGRWO-4") &
    (col("settlementDate") == "2025-06-03") &
    (col("acceptanceNumber") == 20247)
).select(
    "deemedBoFlag",
    "soFlag",
    "storFlag",
    "rrFlag"
).distinct().show(truncate=False)

In [ ]:
# Count combinations of balancing acceptance flags
df_wind_boa.groupBy(
    "deemedBoFlag",
    "soFlag",
    "storFlag",
    "rrFlag"
).count().orderBy(
    "count", ascending=False
).show(truncate=False)

In [ ]:
# Compare power changes by SO Flag
df_wind_changes.groupBy(
    "soFlag",
    "changeType"
).count().orderBy(
    "soFlag", "changeType"
).show()

## 6. Validate record counts, date coverage and repeated segments


In [ ]:
# Count unique wind balancing acceptances
unique_acceptances = df_wind_boa.select(
    "settlementDate",
    "nationalGridBmUnit",
    "acceptanceNumber"
).distinct().count()

print("Total wind BOALF records:", df_wind_boa.count())
print("Unique balancing acceptances:", unique_acceptances)

**Duplicate audit note:** The following check compares a selected seven-column segment key. It does not remove rows or establish that every match is an erroneous duplicate: additional flags or settlement-period information may distinguish records. The original saved Delta table below uses `df_wind_june` without this audit-driven deduplication. If a material repeat count is found, investigate before changing the dataset or quoting KPI counts.


In [ ]:
# Check repeated segments
segment_columns = [
    "settlementDate",
    "nationalGridBmUnit",
    "acceptanceNumber",
    "timeFrom",
    "timeTo",
    "levelFrom",
    "levelTo"
]

print(
    "Repeated segments:",
    df_wind_boa.count() -
    df_wind_boa.select(*segment_columns).distinct().count()
)

In [ ]:
# Count unique wind units in BOALF
print(
    "Wind units with BOALF records:",
    df_wind_boa.select(
        "nationalGridBmUnit"
    ).distinct().count()
)

# Count unique wind units in reference data
print(
    "Wind units in reference data:",
    df_wind.select(
        "nationalGridBmUnit"
    ).distinct().count()
)

In [ ]:
from pyspark.sql.functions import min, max, countDistinct

df_wind_boa.select(
    min("settlementDate").alias("First_Date"),
    max("settlementDate").alias("Last_Date"),
    countDistinct("settlementDate").alias("Total_Days")
).show()

In [ ]:
# Count wind records outside June 2025
df_wind_boa.filter(
    (col("settlementDate") < "2025-06-01") |
    (col("settlementDate") > "2025-06-30")
).groupBy("settlementDate").count().show()

## 7. Filter June BOALF data and save to Delta

**Caution:** This is the original processing logic. It filters settlement dates but does not apply a deduplication operation. `mode("overwrite")` replaces the existing Lakehouse table.


In [ ]:
# Keep only June 2025 settlement dates
df_wind_june = df_wind_boa.filter(
    (col("settlementDate") >= "2025-06-01") &
    (col("settlementDate") <= "2025-06-30")
)

# Check the filtered dataset
print("June wind records:", df_wind_june.count())

In [ ]:
# Save June wind balancing data as a Delta table
df_wind_june.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("wind_boa_june_2025")

print("Delta table saved successfully!")

## 8. Prepare the wind-unit dimension table

The source contains repeated National Grid BM unit IDs in some cases. `dropDuplicates(["nationalGridBmUnit"])` retains one row per ID without a deterministic preference when reference attributes differ. Review such cases before relying on unit attributes.


In [ ]:
# Select important wind unit information
df_wind_reference = df_wind.select(
    "nationalGridBmUnit",
    "elexonBmUnit",
    "bmUnitName",
    "fuelType",
    "generationCapacity"
)

# Remove repeated National Grid unit IDs
df_wind_reference = df_wind_reference.dropDuplicates(
    ["nationalGridBmUnit"]
)

# Check the results
print("Wind reference records:", df_wind_reference.count())

display(df_wind_reference.limit(5))

In [ ]:
from pyspark.sql.functions import col

# Convert generation capacity to a numerical type
df_wind_reference = df_wind_reference.withColumn(
    "generationCapacity",
    col("generationCapacity").cast("double")
)

# Check the updated data types
df_wind_reference.printSchema()

# Count missing generation capacity values
print(
    "Missing generation capacity:",
    df_wind_reference.filter(
        col("generationCapacity").isNull()
    ).count()
)

In [ ]:
# Save wind unit reference data as a Delta table
df_wind_reference.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("wind_unit_reference")

print("Wind reference Delta table saved successfully!")

## 9. Load and validate NESO historic demand

The NESO series is used for generation and demand context. It does not establish the cause of individual wind balancing instructions.


In [ ]:
# Read NESO Historic Demand Data
df_neso = spark.read.csv(
    "Files/NESO Historic Demand Data 2025.csv",
    header=True,
    inferSchema=True
)

# Check columns and data types
df_neso.printSchema()

# Preview the first five rows
display(df_neso.limit(5))

# Count records
print("Total records:", df_neso.count())

In [ ]:
from pyspark.sql.functions import col, sum as spark_sum

# Check missing values in every column
df_neso.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df_neso.columns
]).show(truncate=False)

# Check duplicate rows
print(
    "Duplicate rows:",
    df_neso.count() - df_neso.distinct().count()
)

In [ ]:
# Check missing values in a readable format
from pyspark.sql.functions import col, sum as spark_sum

missing = [
    (c, df_neso.filter(col(c).isNull()).count())
    for c in df_neso.columns
]

for column, count in missing:
    if count > 0:
        print(column, count)

print("Total missing values:", sum(n for _, n in missing))

# Check duplicates
print("Duplicate rows:", df_neso.count() - df_neso.distinct().count())

In [ ]:
# Filter NESO data for June 2025
df_neso_june = df_neso.filter(
    (col("SETTLEMENT_DATE") >= "2025-06-01") &
    (col("SETTLEMENT_DATE") <= "2025-06-30")
)

# Check the number of records
print("June NESO records:", df_neso_june.count())

# Check settlement period coverage
df_neso_june.groupBy("SETTLEMENT_DATE").count() \
    .orderBy("SETTLEMENT_DATE").show(30)

In [ ]:
# Save June 2025 NESO data as a Delta table
df_neso_june.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("neso_demand_june_2025")

print("NESO Delta table saved successfully!")

## 10. Load annual REF wind constraints and save to Delta

The REF monthly dataset provides reported annual cost and curtailed-energy totals, separate from the June Elexon BOALF analysis.


In [ ]:

# Read REF wind constraints data
df_ref = spark.read.csv(
    "Files/REF_Wind_Constraints_2025.csv",
    header=True,
    inferSchema=True
)

# Check the records and data types
df_ref.printSchema()
display(df_ref)


In [ ]:

# Convert Month from timestamp to date
from pyspark.sql.functions import to_date

df_ref_clean = df_ref.withColumn(
    "Month", to_date("Month")
)

# Save as a Delta table in the Lakehouse
df_ref_clean.write.format("delta").mode("overwrite").saveAsTable(
    "ref_wind_constraints_2025"
)

print("REF Delta table saved successfully!")


## 11. Publication notes and limitations

- **BOALF interpretation:** `powerChange < 0` flags downward transitions, not necessarily verified curtailment events. No MWh curtailment estimate is calculated from the BOALF records in this notebook.
- **Repeated segments:** A diagnostic is included; the original pipeline does not remove the detected repeated keys. The downstream KPI counts should be treated as provisional until any material repeats are investigated.
- **Reference table:** Choosing one of several records for a BM unit can affect displayed attributes; inspect collisions before using this as an authoritative dimension.
- **Source coverage:** BOALF and NESO analyses focus on June 2025; REF totals cover 2025. Comparisons are contextual, not causal.
- **Notebook vs dashboard:** Power BI semantic-model relationships, DAX measures, and report visuals are maintained separately from this PySpark notebook.
- **Data access:** Raw datasets are not bundled. Consult the original Elexon, NESO and REF sources and their applicable licences.


## 12. Optional read-only validation checks

Run the cell below **after** the preceding steps, if you want a quick summary of the selected segment-key repetitions and downward transitions. It does not alter tables or dashboard results.


In [ ]:
from pyspark.sql.functions import col, countDistinct

segment_key = [
    "settlementDate", "nationalGridBmUnit", "acceptanceNumber",
    "timeFrom", "timeTo", "levelFrom", "levelTo"
]

print("June wind BOALF rows:", df_wind_june.count())
print("Rows sharing the selected segment key:",
      df_wind_june.count() - df_wind_june.select(*segment_key).distinct().count())
print("June downward power-level segments (before any key deduplication):",
      df_wind_june.filter(col("levelTo") < col("levelFrom")).count())
print("Distinct BM units with June BOALF rows:",
      df_wind_june.select("nationalGridBmUnit").distinct().count())
print("Distinct June settlement dates:",
      df_wind_june.select("settlementDate").distinct().count())
